# 00b · Data probe (first step of checkpoint CP1)

Run every cell from top to bottom in a **fresh CPU runtime** (no GPU units needed). Expected time:
a few minutes up to about half an hour, depending on how many files are on Drive.

It **only reads** `Paper 1/datasets/`: nothing there is written, moved, extracted or re-encoded.
It records what is present and missing against the medium profile, per-folder file formats,
JPEG quality, chroma subsampling, image sizes and 8×8 blockiness (on a seeded sample),
DiffSeg30k mask values, SID-Set label counts, Chameleon counts and the BR-Gen archive parts.
It also downloads the Hugging Face dataset cards (README) of DiffSeg30k, SID-Set and BR-Gen.

Results in `Paper 1/runs/checkpoints/`: `cp1_probe.json` (full record), `cp1_probe_summary.json`
(the same without the folder listing and card texts) and `cp1_probe_cards.json` (card texts).

Secrets: `GITHUB_TOKEN` (read-only) for the clone while the repository is private. `HF_TOKEN` is
read **only if** Hugging Face refuses the card without one; it is never printed or saved.

Before running: if the pull request with this notebook is not merged yet, set
`BRANCH = 'claude/nice-curie-3tzxzr'` in Cell 1. Optionally put the CPU runtime's compute-unit
rate from Colab's resource panel into `CU_PER_HOUR` in Cell 2.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: no Drive, no clone
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        # drop the token from the stored remote URL
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))        # makes the fresh install importable now
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: what this session runs on (no secrets are read or shown)
import json, logging, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import get_paths, session_info

CU_PER_HOUR = None          # optional: the CPU runtime's rate from Colab's resource panel
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
print(json.dumps(info, indent=2))
print('datasets folder:', paths.datasets)

In [ ]:
# Cell 3: probe the datasets folder (read-only)
from eviground.data.probe import format_summary, run_probe
from eviground.env import MissingSecretError, get_secret

if DRY:   # a tiny, partial fake Drive in a temp folder
    from eviground.data.synthetic import make_fake_drive
    if not paths.datasets.exists():
        make_fake_drive(paths.datasets)

def hf_token():
    # called only if Hugging Face refuses a dataset card without a token
    try:
        return get_secret('HF_TOKEN')
    except MissingSecretError:
        return None

report = run_probe(paths.datasets, sample_headers=300, sample_blockiness=40, sample_masks=300,
                   seed=42, fetch_cards=not DRY, token_getter=hf_token)
print(format_summary(report))

In [ ]:
# Cell 4: save the record to Drive (runs/checkpoints/cp1_probe.json)
import datetime

out = paths.runs / 'checkpoints'
out.mkdir(parents=True, exist_ok=True)
hours = (time.time() - T0) / 3600
cards = report.get('hf_cards', {})
diffseg_card = cards.get('Chaos2629/Diffseg30k', {})
record = {
    'checkpoint': 'CP1-probe',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY,
    'branch': BRANCH,
    'session': info,
    'wall_time_h': round(hours, 3),
    'cu_per_hour': CU_PER_HOUR,
    'cu_estimate': round(hours * CU_PER_HOUR, 3) if CU_PER_HOUR else None,
    'diffseg_card_fetched': bool(diffseg_card.get('readme')),
    'probe': report,
}
path = out / 'cp1_probe.json'                     # full record
path.write_text(json.dumps(record, indent=1))
# two small companions, so each stays readable in one piece
summary = {k: v for k, v in record.items() if k != 'probe'}
summary['probe'] = {k: v for k, v in report.items() if k not in ('dirs', 'hf_cards')}
summary['probe']['hf_cards'] = {repo: {k: v for k, v in c.items() if k != 'readme'}
                                for repo, c in cards.items()}
(out / 'cp1_probe_summary.json').write_text(json.dumps(summary, indent=1))
(out / 'cp1_probe_cards.json').write_text(json.dumps(
    {repo: {'readme_status': c.get('readme_status'), 'readme': c.get('readme')}
     for repo, c in cards.items()}, indent=1))
for name in ('cp1_probe.json', 'cp1_probe_summary.json', 'cp1_probe_cards.json'):
    print(f'saved {out / name} ({(out / name).stat().st_size / 1e6:.2f} MB)')
print(f'wall time {hours * 60:.1f} min')
print('DiffSeg30k dataset card fetched:', record['diffseg_card_fetched'],
      '| status:', diffseg_card.get('readme_status', 'not requested'))
assert report['exists'], 'datasets folder not found: is Drive mounted?'